# Food Delivery Rating Prediction using LSTM + Embedding (RNN)

This notebook implements a deep learning model to predict restaurant rating categories from the **Zomato dataset** (India restaurants).

## Architecture Overview

The **ZomatoLSTM** model uses a hybrid approach:

1. **Embedding Layer** — maps each cuisine token (e.g., `north indian`, `chinese`) to a dense vector
2. **LSTM Layer** — processes the cuisine token sequence to capture combination patterns (e.g., *North Indian + Mughlai* signals a different experience than *Italian + Continental*)
3. **Concatenation** — the LSTM's final hidden state is merged with numerical features: price range, votes, average cost for two, online delivery flag, and table booking flag
4. **Classifier Head** — fully connected layers with BatchNorm and Dropout predict one of 5 rating classes

## Target Labels (5-class Classification)

| Label | Class      | Rating Range |
|-------|------------|--------------|
| 0     | Poor       | 0.0 – 2.4    |
| 1     | Average    | 2.5 – 3.4    |
| 2     | Good       | 3.5 – 3.9    |
| 3     | Very Good  | 4.0 – 4.4    |
| 4     | Excellent  | 4.5 – 5.0    |

## Data Sources

- **zomato.csv** — Zomato Kaggle dataset (CSV format, India restaurants filtered by Country Code = 1)
- **file1.json – file5.json** — Zomato API JSON export files with restaurant details and ratings

Both sources are merged, cleaned, and combined into a single training dataset.

In [ ]:
# Install required dependencies
!pip install torch pandas numpy scikit-learn matplotlib seaborn openpyxl

## Step 1: Upload Zomato Data Files

Upload the following files when prompted:

| File | Description |
|------|-------------|
| `zomato.csv` | Zomato Kaggle CSV dataset |
| `file1.json` | Zomato API JSON export (batch 1) |
| `file2.json` | Zomato API JSON export (batch 2) |
| `file3.json` | Zomato API JSON export (batch 3) |
| `file4.json` | Zomato API JSON export (batch 4) |
| `file5.json` | Zomato API JSON export (batch 5) |

> **Note:** All 6 files must be uploaded before proceeding to the next step. The upload cell will confirm each file received.

In [ ]:
from google.colab import files
import os

print("Please upload: zomato.csv, file1.json, file2.json, file3.json, file4.json, file5.json")
uploaded = files.upload()

print("\nUploaded files:")
for fname in uploaded:
    size_kb = len(uploaded[fname]) / 1024
    print(f"  {fname:25s}  {size_kb:8.1f} KB")

# Verify all required files are present
required = ['zomato.csv', 'file1.json', 'file2.json', 'file3.json', 'file4.json', 'file5.json']
missing  = [f for f in required if not os.path.exists(f)]
if missing:
    print(f"\nWARNING: Missing files: {missing}")
else:
    print("\nAll required files confirmed. Ready to proceed.")

## Step 2: Data Preparation

This step performs the following operations:

1. **Load & filter** `zomato.csv` to India-only restaurants (Country Code = 1)
2. **Parse JSON files** and extract restaurant records from the API response structure
3. **Merge** both sources into a single DataFrame
4. **Remove unrated** restaurants (`rating_text == 'Not rated'`)
5. **Tokenize cuisines** — split `"North Indian, Chinese"` into `['north indian', 'chinese']`
6. **Build vocabulary** — assign integer IDs to cuisine tokens (min frequency = 5), with `<PAD>=0` and `<UNK>=1`
7. **Encode sequences** — pad/truncate each restaurant's cuisine list to length 8
8. **Engineer numerical features** — log-transform votes and cost, standardise with StandardScaler
9. **Encode labels** — map rating text to integers 0–4
10. **Split & save** — 80/20 stratified train/test split, save as `.npy` arrays + `metadata.json`

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import re
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# ── Load CSV ──────────────────────────────────────────────────────────────────
print("Loading Zomato CSV...")
csv_df = pd.read_csv('zomato.csv', encoding='latin1')
csv_df = csv_df[csv_df['Country Code'] == 1].copy()
csv_df = csv_df.rename(columns={
    'Cuisines': 'cuisines',
    'Price range': 'price_range',
    'Average Cost for two': 'avg_cost_for_two',
    'Has Online delivery': 'has_online_del',
    'Has Table booking': 'has_table_book',
    'Rating text': 'rating_text',
    'Aggregate rating': 'aggregate_rating',
    'Votes': 'votes',
    'City': 'city',
    'Restaurant Name': 'name',
})
csv_df['has_online_del'] = (csv_df['has_online_del'] == 'Yes').astype(int)
csv_df['has_table_book'] = (csv_df['has_table_book'] == 'Yes').astype(int)
csv_df = csv_df[[
    'cuisines', 'price_range', 'avg_cost_for_two',
    'has_online_del', 'has_table_book', 'city',
    'rating_text', 'aggregate_rating', 'votes', 'name'
]]
print(f"CSV India restaurants: {len(csv_df)}")

# ── Load JSON files ───────────────────────────────────────────────────────────
print("Loading Zomato JSON files...")
json_records = []
for i in range(1, 6):
    fpath = f'file{i}.json'
    with open(fpath, encoding='utf-8') as f:
        data = json.load(f)
    for item in data:
        for r in item.get('restaurants', []):
            rest = r.get('restaurant', {})
            loc  = rest.get('location', {})
            ur   = rest.get('user_rating', {})
            json_records.append({
                'name':             rest.get('name', ''),
                'cuisines':         rest.get('cuisines', ''),
                'price_range':      int(rest.get('price_range', 0)),
                'avg_cost_for_two': float(rest.get('average_cost_for_two', 0)),
                'has_online_del':   int(rest.get('has_online_delivery', 0)),
                'has_table_book':   int(rest.get('has_table_booking', 0)),
                'city':             loc.get('city', ''),
                'rating_text':      ur.get('rating_text', ''),
                'aggregate_rating': float(ur.get('aggregate_rating', 0)),
                'votes':            int(ur.get('votes', 0)),
            })

json_df = pd.DataFrame(json_records)
print(f"JSON India restaurants: {len(json_df)}")

# ── Merge & clean ─────────────────────────────────────────────────────────────
df = pd.concat([csv_df, json_df], ignore_index=True)
print(f"Combined total: {len(df)}")

df = df[df['rating_text'].str.strip() != 'Not rated']
df = df[df['rating_text'].notna() & (df['rating_text'].str.strip() != '')]
df = df[df['cuisines'].notna() & (df['cuisines'].str.strip() != '')]
df = df.reset_index(drop=True)
print(f"After removing 'Not rated': {len(df)}")
print("Rating distribution:\n", df['rating_text'].value_counts())

# ── Cuisine tokenisation ──────────────────────────────────────────────────────
def parse_cuisines(c_str):
    return [t.strip().lower() for t in str(c_str).split(',') if t.strip()]

df['cuisine_tokens'] = df['cuisines'].apply(parse_cuisines)

all_tokens = [tok for tokens in df['cuisine_tokens'] for tok in tokens]
freq = Counter(all_tokens)
print(f"\nUnique cuisine types: {len(freq)}")
print("Top 15 cuisines:", [t for t, _ in freq.most_common(15)])

# ── Build vocabulary ──────────────────────────────────────────────────────────
MIN_FREQ  = 5
vocab     = ['<PAD>', '<UNK>'] + [tok for tok, cnt in freq.most_common() if cnt >= MIN_FREQ]
vocab_size = len(vocab)
tok2idx   = {tok: idx for idx, tok in enumerate(vocab)}
print(f"Vocabulary size (min_freq={MIN_FREQ}): {vocab_size}")

# ── Encode sequences ──────────────────────────────────────────────────────────
MAX_SEQ_LEN = 8

def encode_sequence(tokens):
    ids = [tok2idx.get(tok, 1) for tok in tokens]
    if len(ids) < MAX_SEQ_LEN:
        ids = ids + [0] * (MAX_SEQ_LEN - len(ids))
    else:
        ids = ids[:MAX_SEQ_LEN]
    return ids

df['cuisine_ids'] = df['cuisine_tokens'].apply(encode_sequence)

# ── Numerical features ────────────────────────────────────────────────────────
df['votes']            = pd.to_numeric(df['votes'],            errors='coerce').fillna(0)
df['avg_cost_for_two'] = pd.to_numeric(df['avg_cost_for_two'], errors='coerce').fillna(500)
df['price_range']      = pd.to_numeric(df['price_range'],      errors='coerce').fillna(2)

df['log_votes'] = np.log1p(df['votes'])
df['log_cost']  = np.log1p(df['avg_cost_for_two'])

NUM_FEATURES = ['price_range', 'has_online_del', 'has_table_book', 'log_votes', 'log_cost']
df[NUM_FEATURES] = df[NUM_FEATURES].fillna(0)

scaler = StandardScaler()

# ── Label encoding ────────────────────────────────────────────────────────────
RATING_MAP = {'Poor': 0, 'Average': 1, 'Good': 2, 'Very Good': 3, 'Excellent': 4}
df['label'] = df['rating_text'].str.strip().map(RATING_MAP)
df = df[df['label'].notna()].reset_index(drop=True)
df['label'] = df['label'].astype(int)

print(f"\nFinal dataset size: {len(df)}")
print("Label distribution:\n", df['label'].value_counts().sort_index())

# ── Arrays ────────────────────────────────────────────────────────────────────
X_seq = np.array(df['cuisine_ids'].tolist(), dtype=np.int64)
X_num = df[NUM_FEATURES].values.astype(np.float32)
y     = df['label'].values.astype(np.int64)

X_seq_tr, X_seq_te, X_num_tr, X_num_te, y_tr, y_te = train_test_split(
    X_seq, X_num, y, test_size=0.2, stratify=y, random_state=42
)

X_num_tr = scaler.fit_transform(X_num_tr).astype(np.float32)
X_num_te = scaler.transform(X_num_te).astype(np.float32)

print(f"\nTrain: {X_seq_tr.shape[0]:,}  |  Test: {X_seq_te.shape[0]:,}")

# ── Save processed data ───────────────────────────────────────────────────────
os.makedirs('data/processed', exist_ok=True)
np.save('data/processed/X_seq_train.npy', X_seq_tr)
np.save('data/processed/X_seq_test.npy',  X_seq_te)
np.save('data/processed/X_num_train.npy', X_num_tr)
np.save('data/processed/X_num_test.npy',  X_num_te)
np.save('data/processed/y_train.npy',     y_tr)
np.save('data/processed/y_test.npy',      y_te)

meta = {
    'vocab_size':   vocab_size,
    'vocab':        vocab[:200],
    'max_seq_len':  MAX_SEQ_LEN,
    'num_features': NUM_FEATURES,
    'num_classes':  5,
    'class_names':  ['Poor', 'Average', 'Good', 'Very Good', 'Excellent'],
}
with open('data/processed/metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)

print("Data preparation complete.")

## Step 3: Build and Train LSTM + Embedding Model

### Model Architecture: ZomatoLSTM

```
Input: cuisine_ids (seq_len=8)     Input: numerical features (5)
          │                                      │
   Embedding(vocab→32)                           │
          │                                      │
   LSTM(32→128, 2 layers)                        │
          │                                      │
   last hidden state (128)  ──── concat ─────────┘
                                   │  (128+5=133)
                             Dropout(0.3)
                             Linear(133→128)
                             BatchNorm1d + ReLU
                             Dropout(0.2)
                             Linear(128→64)
                             BatchNorm1d + ReLU
                             Linear(64→5)
                                   │
                             Logits (5 classes)
```

### Training Configuration

| Hyperparameter | Value |
|----------------|-------|
| Embedding dim  | 32    |
| LSTM hidden    | 128   |
| LSTM layers    | 2     |
| Dropout        | 0.3   |
| Batch size     | 128   |
| Epochs         | 30    |
| Optimizer      | Adam (lr=1e-3, weight_decay=1e-5) |
| LR Scheduler   | StepLR (step=8, gamma=0.5) |
| Loss           | CrossEntropyLoss with class weights |
| Grad clip      | max_norm=1.0 |

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import os
import json
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print("PyTorch version:", torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ── Load processed data ───────────────────────────────────────────────────────
X_seq_train = np.load('data/processed/X_seq_train.npy')
X_seq_test  = np.load('data/processed/X_seq_test.npy')
X_num_train = np.load('data/processed/X_num_train.npy')
X_num_test  = np.load('data/processed/X_num_test.npy')
y_train     = np.load('data/processed/y_train.npy')
y_test      = np.load('data/processed/y_test.npy')

with open('data/processed/metadata.json') as f:
    meta = json.load(f)

VOCAB_SIZE   = meta['vocab_size']
MAX_SEQ_LEN  = meta['max_seq_len']
NUM_FEATURES = len(meta['num_features'])
NUM_CLASSES  = meta['num_classes']
CLASS_NAMES  = meta['class_names']

print(f"Train: {X_seq_train.shape[0]:,}  |  Test: {X_seq_test.shape[0]:,}")

# ── DataLoaders ───────────────────────────────────────────────────────────────
BATCH_SIZE = 128

def make_loader(X_seq, X_num, y, shuffle=False):
    ds = TensorDataset(
        torch.tensor(X_seq, dtype=torch.long),
        torch.tensor(X_num, dtype=torch.float32),
        torch.tensor(y,     dtype=torch.long),
    )
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

train_loader = make_loader(X_seq_train, X_num_train, y_train, shuffle=True)
test_loader  = make_loader(X_seq_test,  X_num_test,  y_test,  shuffle=False)

# ── Model definition ──────────────────────────────────────────────────────────
class ZomatoLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_feat,
                 num_classes, num_lstm_layers=2, dropout=0.3):
        super(ZomatoLSTM, self).__init__()
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=0
        )
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_lstm_layers,
            batch_first=True,
            dropout=dropout if num_lstm_layers > 1 else 0.0
        )
        combined_dim = hidden_dim + num_feat
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(combined_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x_seq, x_num):
        embedded = self.embedding(x_seq)
        lstm_out, (h_n, _) = self.lstm(embedded)
        last_hidden = h_n[-1]
        combined = torch.cat([last_hidden, x_num], dim=1)
        return self.classifier(combined)

model = ZomatoLSTM(
    vocab_size=VOCAB_SIZE,
    embed_dim=32,
    hidden_dim=128,
    num_feat=NUM_FEATURES,
    num_classes=NUM_CLASSES,
    num_lstm_layers=2,
    dropout=0.3
).to(device)

print(f"\nModel:\n{model}")
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")

# ── Class-weighted loss ───────────────────────────────────────────────────────
class_counts = np.bincount(y_train, minlength=NUM_CLASSES).astype(float)
total = class_counts.sum()
class_weights = torch.tensor(
    [total / (NUM_CLASSES * c) if c > 0 else 1.0 for c in class_counts],
    dtype=torch.float32
).to(device)
print(f"\nClass weights: {class_weights.tolist()}")

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=8, gamma=0.5)

# ── Training loop ─────────────────────────────────────────────────────────────
NUM_EPOCHS = 30
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
print(f"\nStarting training for {NUM_EPOCHS} epochs...")

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    for X_s, X_n, y_b in train_loader:
        X_s, X_n, y_b = X_s.to(device), X_n.to(device), y_b.to(device)
        optimizer.zero_grad()
        outputs = model(X_s, X_n)
        loss    = criterion(outputs, y_b)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += loss.item() * X_s.size(0)
    train_loss = running_loss / len(train_loader.dataset)
    history['train_loss'].append(train_loss)

    model.eval()
    val_loss, correct, total_val = 0.0, 0, 0
    with torch.no_grad():
        for X_s, X_n, y_b in test_loader:
            X_s, X_n, y_b = X_s.to(device), X_n.to(device), y_b.to(device)
            out  = model(X_s, X_n)
            loss = criterion(out, y_b)
            val_loss += loss.item() * X_s.size(0)
            preds = out.argmax(dim=1)
            correct   += (preds == y_b).sum().item()
            total_val += y_b.size(0)
    val_loss /= len(test_loader.dataset)
    val_acc   = correct / total_val
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    scheduler.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1:2d}/{NUM_EPOCHS}]  "
              f"Train Loss: {train_loss:.4f}  "
              f"Val Loss: {val_loss:.4f}  "
              f"Val Acc: {val_acc:.4f}")

# ── Evaluation ────────────────────────────────────────────────────────────────
print("\nEvaluating on test set...")
model.eval()
all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for X_s, X_n, y_b in test_loader:
        X_s, X_n = X_s.to(device), X_n.to(device)
        out   = model(X_s, X_n)
        probs = torch.softmax(out, dim=1)
        preds = out.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_b.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

accuracy  = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
recall    = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
f1        = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
cm        = confusion_matrix(all_labels, all_preds)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")
print(f"\nConfusion Matrix ({CLASS_NAMES}):")
print(cm)

# ── Save model and metrics ────────────────────────────────────────────────────
os.makedirs('saved_models', exist_ok=True)
torch.save(model.state_dict(), 'saved_models/lstm_model.pth')

metrics = {
    'accuracy':         round(accuracy,  4),
    'precision':        round(precision, 4),
    'recall':           round(recall,    4),
    'f1':               round(f1,        4),
    'confusion_matrix': cm.tolist(),
    'class_names':      CLASS_NAMES,
    'history':          history,
    'vocab_size':       VOCAB_SIZE,
}
with open('saved_models/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Model and metrics saved.")

## Step 4: Visualize Results

This step generates five plots:

1. **Training & Validation Loss Curves** — monitors overfitting across 30 epochs
2. **Validation Accuracy per Epoch** — tracks learning progress with final test accuracy reference line
3. **Confusion Matrix (Counts)** — absolute prediction counts per class pair
4. **Confusion Matrix (Normalised)** — per-class recall as proportions (highlights class imbalance effects)
5. **Overall Metric Bars** — Accuracy, Precision, Recall, F1 on the test set
6. **Per-Class Precision, Recall & F1** — grouped bar chart for each of the 5 rating categories
7. **Rating Distribution Pie** — proportion of each rating class in the full dataset

All plots are saved to `visualizations/` and displayed inline.

In [ ]:
%matplotlib inline
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score

os.makedirs('visualizations', exist_ok=True)

with open('saved_models/metrics.json') as f:
    metrics = json.load(f)

CLASS_NAMES = metrics['class_names']

# ── Plot 1 & 2: Loss and accuracy curves ──────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
epochs = range(1, len(metrics['history']['train_loss']) + 1)

ax1.plot(epochs, metrics['history']['train_loss'], 'b-o', markersize=3, label='Train Loss')
ax1.plot(epochs, metrics['history']['val_loss'],   'r-s', markersize=3, label='Val Loss')
ax1.set_title('Training & Validation Loss', fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(epochs, [v * 100 for v in metrics['history']['val_acc']], 'g-^', markersize=3, label='Val Accuracy')
ax2.axhline(
    metrics['accuracy'] * 100,
    color='orange', linestyle='--',
    label=f"Final Test Acc: {metrics['accuracy']*100:.1f}%"
)
ax2.set_title('Validation Accuracy per Epoch', fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle(
    'LSTM Cuisine-Sequence Model — Training Curves\nFood Delivery Rating Prediction',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('visualizations/loss_curve.png', dpi=150)
plt.show()

# ── Plot 3 & 4: Confusion matrices ────────────────────────────────────────────
cm      = np.array(metrics['confusion_matrix'])
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0]
)
axes[0].set_title('Confusion Matrix (Counts)', fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

sns.heatmap(
    cm_norm, annot=True, fmt='.2f', cmap='YlOrRd',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1]
)
axes[1].set_title('Confusion Matrix (Normalised)', fontweight='bold')
axes[1].set_ylabel('Actual')
axes[1].set_xlabel('Predicted')

plt.suptitle(
    'LSTM Restaurant Rating Classifier — Confusion Matrices',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig('visualizations/confusion_matrix.png', dpi=150)
plt.show()

# ── Plot 5: Overall metric bars ───────────────────────────────────────────────
metric_names  = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
metric_values = [metrics['accuracy'], metrics['precision'], metrics['recall'], metrics['f1']]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    metric_names, metric_values,
    color=['#2196F3', '#4CAF50', '#FF9800', '#9C27B0'],
    width=0.5, edgecolor='white', linewidth=1.2
)
for bar, val in zip(bars, metric_values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.005,
        f'{val:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold'
    )
ax.set_ylim(0, 1.12)
ax.set_title(
    'LSTM Rating Predictor — Test Set Performance\nFood Delivery (Zomato)',
    fontsize=12, fontweight='bold'
)
ax.set_ylabel('Score')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('visualizations/metric_bars.png', dpi=150)
plt.show()

# ── Plot 6: Per-class metrics (reload model for fresh predictions) ─────────────
class ZomatoLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_feat,
                 num_classes, num_lstm_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, num_lstm_layers,
            batch_first=True,
            dropout=dropout if num_lstm_layers > 1 else 0.0
        )
        combined_dim = hidden_dim + num_feat
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(combined_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x_seq, x_num):
        emb = self.embedding(x_seq)
        _, (h_n, _) = self.lstm(emb)
        return self.classifier(torch.cat([h_n[-1], x_num], dim=1))

with open('data/processed/metadata.json') as f:
    meta = json.load(f)

mdl = ZomatoLSTM(meta['vocab_size'], 32, 128, len(meta['num_features']), 5)
mdl.load_state_dict(torch.load('saved_models/lstm_model.pth', weights_only=True))
mdl.eval()

X_s    = torch.tensor(np.load('data/processed/X_seq_test.npy'), dtype=torch.long)
X_n    = torch.tensor(np.load('data/processed/X_num_test.npy'), dtype=torch.float32)
y_t    = torch.tensor(np.load('data/processed/y_test.npy'),     dtype=torch.long)
loader = DataLoader(TensorDataset(X_s, X_n, y_t), batch_size=256, shuffle=False)

preds_all = []
with torch.no_grad():
    for xs, xn, _ in loader:
        preds_all.extend(mdl(xs, xn).argmax(dim=1).numpy())
preds_all = np.array(preds_all)
y_test_np = np.load('data/processed/y_test.npy')

prec_per = precision_score(y_test_np, preds_all, average=None, zero_division=0)
rec_per  = recall_score(y_test_np,    preds_all, average=None, zero_division=0)
f1_per   = f1_score(y_test_np,        preds_all, average=None, zero_division=0)

x = np.arange(len(CLASS_NAMES))
w = 0.25
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w, prec_per, w, label='Precision', color='steelblue')
ax.bar(x,     rec_per,  w, label='Recall',    color='darkorange')
ax.bar(x + w, f1_per,   w, label='F1-Score',  color='green')
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES)
ax.set_ylim(0, 1.15)
ax.set_title(
    'Per-Class Precision, Recall & F1 — LSTM Zomato Rating Predictor',
    fontweight='bold'
)
ax.set_ylabel('Score')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('visualizations/per_class_metrics.png', dpi=150)
plt.show()

# ── Plot 7: Rating distribution pie ──────────────────────────────────────────
label_counts = (
    np.bincount(np.load('data/processed/y_train.npy'), minlength=5) +
    np.bincount(y_test_np, minlength=5)
)
colors_pie = ['#e74c3c', '#e67e22', '#3498db', '#2ecc71', '#9b59b6']

fig, ax = plt.subplots(figsize=(7, 6))
wedges, texts, autotexts = ax.pie(
    label_counts,
    labels=CLASS_NAMES,
    autopct='%1.1f%%',
    colors=colors_pie,
    startangle=140,
    wedgeprops=dict(edgecolor='white', linewidth=1.5)
)
for at in autotexts:
    at.set_fontsize(10)
ax.set_title(
    'Zomato Restaurant Rating Distribution\n(India, 2019-2023)',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig('visualizations/rating_distribution.png', dpi=150)
plt.show()

print("All visualizations saved to visualizations/")